# Topico 2: Desigualdade de Processamento de Dados e Cadeias de Markov

Cenario: telefone sem fio / cadeia de canais ruidosos. Enunciado: `propostas/1_Simulacoes.pdf`, secao 2.

## Preparacao

In [ ]:
import sys, pathlib

REPO = pathlib.Path.cwd().parents[2]          # .../<topico>/ -> .../<proposta>/ -> raiz
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from utils.info_theory import (
    entropy, binary_entropy, joint_entropy, conditional_entropy,
    cross_entropy, kl_divergence, mutual_information,
    conditional_mutual_information, mi_gtest_statistic, counts_to_joint,
)
from utils.simulation import (
    rng, sample_pmf, sample_joint, empirical_pmf, bsc, markov_chain,
)
from utils.estimation import bootstrap_ci
from utils.plotting import setup_estilo, salvar_fig, plot_pmf, tabela, plot_curvas, plot_estimativa_vs_n
import src.model as model                     # helpers especificos deste topico

setup_estilo()
GEN = rng(20260902)                           # seed fixo -> resultados reproduziveis
DADOS = pathlib.Path("dados")
FIGURAS = pathlib.Path("figuras")

## Passo 1: Fonte X

Definir a variavel fonte X (um bit ou um pequeno vetor de bits) e sua distribuicao.

In [ ]:
# Passo 1
P_X = np.array([0.5, 0.5])  # P(X=0), P(X=1): bit fonte da mensagem original

H_X = entropy(P_X)
print(f"P(X) = {P_X}")
print(f"H(X) = {H_X:.4f} bits  (teto de I(X; Y_j) ao longo da cadeia)")

plot_pmf(P_X, rotulos=["0", "1"], ylabel="P(X)")
salvar_fig("t2_passo1_fonte_x")


## Passo 2: Cadeia de canais BSC

Modelar cada elo como um BSC de parametro p; gerar X -> Y1 -> ... -> Yk com `model.compose_bsc_chain` (guardar TODOS os estagios).

In [ ]:
# Passo 2
K = 5                 # numero de elos da cadeia (Y1, ..., Y5)
P_LIST = [0.10] * K   # cada elo eh um BSC com esse parametro p (mesmo p em todos os elos por ora)

x_demo = np.array([0, 1, 1, 0, 1])
estagios_demo = model.compose_bsc_chain(x_demo, P_LIST, GEN)
for j, estagio in enumerate(estagios_demo):
    nome = "X" if j == 0 else f"Y{j}"
    print(f"{nome} = {estagio}")


## Passo 3: Propagar N mensagens

Simular a cadeia para N mensagens independentes.

In [ ]:
# Passo 3
N = 20000  # tamanho da amostra usada para estimar I(X; Yj) ao longo da cadeia (Passo 4)

X_N = sample_pmf(P_X, N, GEN)
ESTAGIOS = model.compose_bsc_chain(X_N, P_LIST, GEN)  # [X, Y1, ..., YK], cada um com N mensagens
print(f"N = {N} mensagens propagadas por {K} elos")
for j, estagio in enumerate(ESTAGIOS):
    nome = "X" if j == 0 else f"Y{j}"
    print(f"P({nome}=1) empirico = {estagio.mean():.4f}")


## Passo 4: I(X; Yj) para cada j

Estimar I(X; Y1), ..., I(X; Yk) e verificar que a sequencia e aproximadamente nao crescente (dentro da margem de estimacao).

In [ ]:
# Passo 4
def _i_xy(x, y):
    return mutual_information(counts_to_joint(x, y, size_x=2, size_y=2))


I_ESTIMADO, I_IC_BAIXO, I_IC_ALTO = [], [], []
for Y_j in ESTAGIOS[1:]:
    est, baixo, alto = bootstrap_ci((X_N, Y_j), _i_xy, generator=GEN, paired=True)
    I_ESTIMADO.append(est)
    I_IC_BAIXO.append(baixo)
    I_IC_ALTO.append(alto)

TABELA_I = tabela(
    {
        "elo": [f"I(X;Y{j})" for j in range(1, K + 1)],
        "estimado": I_ESTIMADO,
        "IC_baixo": I_IC_BAIXO,
        "IC_alto": I_IC_ALTO,
    },
    indice="elo",
)
display(TABELA_I)

DIFF_I = np.diff(I_ESTIMADO)
print(f"H(X) = {H_X:.4f} bits  (teto de I(X; Yj) ao longo da cadeia)")
print(f"I(X;Yj) - I(X;Yj-1): {np.round(DIFF_I, 4)}")
print(f"sequencia aproximadamente nao crescente: {bool(np.all(DIFF_I <= 0.01))}")


## Passo 5: Grafico da degradacao

Plotar I(X; Yj) vs posicao j na cadeia.

In [ ]:
# Passo 5
J_LIST = list(range(1, K + 1))

plot_estimativa_vs_n(
    J_LIST, I_ESTIMADO, I_IC_BAIXO, I_IC_ALTO,
    teorico=H_X, label="I(X;Yj) estimado",
    xlabel="posicao j na cadeia (Yj)", ylabel="bits", logx=False,
)
salvar_fig("t2_passo5_degradacao_i")


## Passo 6: Variar o ruido

Repetir com elos mais/menos ruidosos e observar a velocidade de degradacao.

In [ ]:
# Passo 6


## Passo 7: Cadeia inspirada em IA

X = rotulo verdadeiro; gerar imagem (`model.class_conditional_gaussian`), comprimir (menos coordenadas), decidir. Estimar I(rotulo; cada estagio) e checar a DPI.

In [ ]:
# Passo 7


## Passo 8: Colapso de modelo

Simular varias geracoes: a cada geracao, estimar uma distribuicao a partir de uma amostra finita da anterior. Medir I ou D(p_real || p_ger k) ao longo das geracoes.

In [ ]:
# Passo 8


## Relatorio

Consolidar resultados, tabelas e figuras em `relatorio.md` (figuras salvas em `figuras/`).